In [1]:
"""
convert_race_export.py
----------------------
Converts the wide-format Nemo/race export Excel file into a flat
tabular CSV (one row per race).
 
Usage:
    python convert_race_export.py <input.xlsx> <output.csv>
 
If no arguments are given, defaults to:
    input:  race_export.xlsx
    output: race_data_converted.csv
"""

'\nconvert_race_export.py\n----------------------\nConverts the wide-format Nemo/race export Excel file into a flat\ntabular CSV (one row per race).\n\nUsage:\n    python convert_race_export.py <input.xlsx> <output.csv>\n\nIf no arguments are given, defaults to:\n    input:  race_export.xlsx\n    output: race_data_converted.csv\n'

In [2]:
import sys
import re
import openpyxl
import pandas as pd

In [3]:
#Helpers

def get(ws, row, col):
    """Safe cell read."""
    v = ws.cell(row=row, column=col).value
    return v
 
 
def parse_time(t):
    """Convert time string (MM:SS.ss or HH:MM:SS) or numeric to seconds."""
    if t is None:
        return None
    if isinstance(t, (int, float)):
        return float(t)
    t = str(t).strip()
    #MM:SS.ss to seconds
    m = re.match(r'^(\d+):(\d+\.\d+)$', t)
    if m:
        return int(m.group(1)) * 60 + float(m.group(2))
    try:
        return float(t)
    except (ValueError, TypeError):
        return None
 
 
def parse_header(h):
    """
    Parse the race header string:
        'Athlete Name DD/MM/YYYY [optional (N)] Distance m Stroke Phase'
 
    Returns a dict of metadata fields.
    """
    if not h:
        return {}
    h = str(h).strip()
 
    #Extract date
    date_m = re.search(r'(\d{2}/\d{2}/\d{4})', h)
    date_str = date_m.group(1) if date_m else None
 
    #Athlete name (Surname) = everything before the date
    athlete = h[:date_m.start()].strip() if date_m else None
 
    #Everything after the date
    after_date = h[date_m.end():].strip() if date_m else h
 
    #Remove optional duplicate-race marker e.g. "(1)"
    after_date = re.sub(r'^\(\d+\)\s*', '', after_date)
 
    #Phase = last word  (Heat/Final/Semi-Final etc.)
    parts = after_date.rsplit(' ', 1)
    phase = parts[1] if len(parts) > 1 else None
    event = parts[0].strip() if len(parts) > 1 else after_date
 
    #Distance + stroke
    event_m = re.match(r'(\d+(?:\s*x\s*\d+)?)\s*m\s+(.*)', event)
    distance = event_m.group(1).strip() if event_m else None
    stroke   = event_m.group(2).strip() if event_m else event
 
    is_relay = bool(re.search(r'4\s*x', event, re.I)) or 'Relay' in stroke
    is_mixed = 'Mixed' in stroke
 
    #Pool length: assume 50 m (LCM) by default — update if you have this info
    pool_length = 50
 
    return {
        'AthleteName':              athlete,
        'AthleteFirstName':         athlete.split()[0] if athlete else None,
        'AthleteLastName':          ' '.join(athlete.split()[1:]) if athlete else None,
        'Date':                     date_str,
        'RaceEventDistance':        distance,
        'RaceEventSwimmingStyleName': stroke,
        'PhaseName':                phase,
        'RaceEventIsRelay':         is_relay,
        'RaceEventIsMixed':         is_mixed,
        'RaceEventName':            f"{distance} m {stroke}" if distance else stroke,
        'PoolLength':               pool_length,
        #Fields we can't derive from the export (left blank for manual fill or keep null)
        'RaceId':                   None,
        'PhaseNumber':              None,
        'Lane':                     None,
        'RelayLeg':                 None,
        'FinishingPosition':        None,
        'RaceEventId':              None,
        'RaceEventSwimmingStyleId': None,
        'AthleteId':                None,
        'AthleteGenderId':          None,
        'AthleteGenderName':        None,
        'AthleteNationId':          None,
        'AthleteNationCode':        None,
        'AthleteNationName':        None,
        'CompetitionId':            None,
        'CompetitionName':          None,
        'PhaseId':                  None,
        'SessionId':                None,
        'SessionCode':              None,
        'AnalysisLevelId':          None,
        'AnalysisLevelName':        None,
    }
 
 

In [13]:
#Main parser
 
def parse_workbook(path):
    wb = openpyxl.load_workbook(path)
    ws = wb.active
 
    #Locate race blocks: row 1, any column whose value contains a date string
    race_cols = []
    for col in range(2, ws.max_column + 1):
        val = ws.cell(row=1, column=col).value
        if val and isinstance(val, str) and re.search(r'\d{2}/\d{2}/\d{4}', val):
            race_cols.append(col)
 
    print(f"Found {len(race_cols)} races.")
 
    records = []
    for base_col in race_cols:
        header = get(ws, 1, base_col)
        meta   = parse_header(header)
 
        #Split times (rows 3–5, col A = label, base_col = time, base_col+1 = %) 
        splits = {}
        for r in range(3, 6):
            label = get(ws, r, 1)
            time_val = parse_time(get(ws, r, base_col))
            if label and time_val is not None:
                #Clean label: "0 - 50 m" → "Split_0_50m"
                clean = re.sub(r'[\s\-]+', '_', str(label)).replace('__', '_').strip('_')
                splits[f'Split_{clean}'] = time_val

        # ── Total race time (row 6) ───────────────────────────────────────────
        total_time = parse_time(get(ws, 6, base_col))
 
        # ── Start metrics (rows 9–13) ─────────────────────────────────────────
        #   Row 9:  START BLOCK (s)
        #   Row 10: START BREAK (m)  = breakout distance
        #   Row 11: START BREAK (s)  = breakout time
        #   Row 12: START 15 m (s)
        #   Row 13: START KICKS
        start = {
            'StartBlockElapsedTime':  get(ws, 9,  base_col),
            'StartBreakoutDistance':  get(ws, 10, base_col),
            'StartBreakoutTime':   get(ws, 11, base_col),
            'Lap1_15mElapsedTime': get(ws, 12, base_col),  
            'StartKicks':      get(ws, 13, base_col),
        }

        # ── Turn metrics (rows 16–21) ─────────────────────────────────────────
        # Column base_col = lap number, base_col+1 = value
        turn = {
            'Turn_5mIn_s':            get(ws, 16, base_col + 1),
            'Turn1RotateTime':        get(ws, 17, base_col + 1),
            'Turn1_15m_s':            get(ws, 18, base_col + 1),
            'Turn1BreakoutDistance':  get(ws, 19, base_col + 1),
            'Turn1BreakoutTime':      get(ws, 20, base_col + 1),
            'Turn1_Total_s':          get(ws, 21, base_col + 1),
        }
 
        # ── Last 5 m (row 24) ─────────────────────────────────────────────────
        last5 = parse_time(get(ws, 24, base_col))

        # ── Stroke rate section (rows 28–31) ──────────────────────────────────
        # Columns: base_col = T, +1 = SR, +2 = DPS, +3 = V, +4 = #
        sr_zones = {28: '15_25m', 29: '25_50m', 30: '65_75m', 31: '75_100m'}
        stroke_data = {}
        for r, zone in sr_zones.items():
            stroke_data[f'SR_{zone}_T']   = get(ws, r, base_col)
            stroke_data[f'SR_{zone}_SR']  = get(ws, r, base_col + 1)
            stroke_data[f'SR_{zone}_DPS'] = get(ws, r, base_col + 2)
            stroke_data[f'SR_{zone}_V']   = get(ws, r, base_col + 3)
            stroke_data[f'SR_{zone}_N']   = get(ws, r, base_col + 4)
 
        row = {
            **meta,
            'RaceTime':  total_time,
            **splits,
            **start,
            **turn,
            'Last5m_s':  last5,
            **stroke_data,
        }
        records.append(row)
 
    return pd.DataFrame(records)
 
 

In [15]:
#Entry point
 
input_file  = '2026 Nemo.xlsx'
output_file = 'race_data_converted.csv'

df = parse_workbook(input_file)
df.to_csv(output_file, index=False)
print(f"Saved: {output_file}")
print(f"Shape: {df.shape}")
print(df[['AthleteName', 'Date', 'RaceEventName', 'PhaseName', 'RaceTime']].head())

Found 394 races.
Saved: race_data_converted.csv
Shape: (394, 66)
  AthleteName        Date                    RaceEventName PhaseName  RaceTime
0       Fearn  08/07/2026  4 x 100 m Freestyle Mixed Relay      Heat     50.16
1       Fearn  07/07/2026        4 x 100 m Freestyle Relay     Final     49.27
2       Fearn  07/07/2026        4 x 100 m Freestyle Relay      Heat     50.24
3      Grieve  28/06/2026                  100 m Butterfly      Heat     59.37
4      Grieve  28/06/2026                  100 m Butterfly     Final     58.64
